# Q14: whole-recording spike-network modularity plots

This notebook contains visualization only. Modularity is calculated by `q14_modularity.py`, and mouse-level statistics are calculated by `q14_modularity_stats.R`.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

repository = Path(os.environ.get('CON2PHYS_REPOSITORY_DIR', 'D:/git_repos/con2phys_submission'))
output_dir = repository / 'results' / 'q14'
mouse_area = pd.read_csv(output_dir / 'q14_modularity_mouse_area.csv')
area_means = pd.read_csv(output_dir / 'q14_modularity_stats_means.csv')
sensitivity = pd.read_csv(output_dir / 'q14_modularity_sensitivity_mouse_area.csv')
unit_counts = pd.read_csv(output_dir / 'q14_unit_counts.csv')

area_labels = {1: 'Brain area 1', 2: 'Brain area 2', 3: 'Brain area 3'}
palette = {1: '#4477AA', 2: '#EE6677', 3: '#228833'}
mouse_area['brain_area_label'] = mouse_area['brain_area'].map(area_labels)
sensitivity['brain_area_label'] = sensitivity['brain_area'].map(area_labels)
sns.set_theme(style='ticks', context='talk')
mouse_area.head()

## Paired mouse-level values

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.5))
ordered = mouse_area.sort_values(['mouse_id', 'brain_area'])
for _, group in ordered.groupby('mouse_id'):
    ax.plot(group['brain_area'], group['mean_delta_q'], color='0.72', linewidth=1, alpha=0.75, zorder=1)
for area in (1, 2, 3):
    values = ordered.loc[ordered['brain_area'].eq(area)]
    ax.scatter(values['brain_area'], values['mean_delta_q'], s=42, color=palette[area], edgecolor='white', linewidth=0.5, zorder=2, label=area_labels[area])
ax.axhline(0, color='0.35', linestyle='--', linewidth=1)
ax.set_xticks([1, 2, 3], [area_labels[i] for i in (1, 2, 3)])
ax.set_xlabel('')
ax.set_ylabel(r'Chance-corrected modularity ($\Delta Q$)')
ax.set_title('Whole-recording spike-network modularity')
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(output_dir / 'q14_modularity_mouse_paired.png', dpi=200, bbox_inches='tight')
plt.show()

## Model-estimated means and 95% confidence intervals

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 5.2))
x = np.arange(len(area_means))
means = area_means['emmean'].to_numpy()
lower = means - area_means['lower.CL'].to_numpy()
upper = area_means['upper.CL'].to_numpy() - means
colors = [palette[int(area)] for area in area_means['brain_area']]
ax.errorbar(x, means, yerr=np.vstack([lower, upper]), fmt='none', ecolor='0.2', elinewidth=1.5, capsize=5)
ax.scatter(x, means, s=90, c=colors, edgecolor='white', linewidth=0.7, zorder=3)
ax.axhline(0, color='0.35', linestyle='--', linewidth=1)
ax.set_xticks(x, area_means['brain_area_label'])
ax.set_xlabel('')
ax.set_ylabel(r'Model-estimated mean $\Delta Q$')
ax.set_title('Mean and 95% confidence interval')
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(output_dir / 'q14_modularity_means_95ci.png', dpi=200, bbox_inches='tight')
plt.show()

## Observed and degree-preserving null modularity

In [ ]:
long_q = mouse_area.melt(id_vars=['mouse_id', 'brain_area', 'brain_area_label'], value_vars=['mean_q_observed', 'mean_q_null'], var_name='network', value_name='Q')
long_q['network'] = long_q['network'].map({'mean_q_observed': 'Observed', 'mean_q_null': 'Degree-preserving null'})
fig, ax = plt.subplots(figsize=(8.5, 5.5))
sns.boxplot(data=long_q, x='brain_area_label', y='Q', hue='network', showfliers=False, palette=['#4C78A8', '#B8B8B8'], ax=ax)
sns.stripplot(data=long_q, x='brain_area_label', y='Q', hue='network', dodge=True, color='0.2', size=3, alpha=0.55, legend=False, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('Optimized modularity (Q)')
ax.set_title('Observed networks versus degree-preserving nulls')
ax.legend(title='Network', frameon=False)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(output_dir / 'q14_modularity_observed_vs_null.png', dpi=200, bbox_inches='tight')
plt.show()

## Density and resolution sensitivity

In [ ]:
density_data = sensitivity.loc[(sensitivity['setting_type'].isin(['density', 'primary'])) & sensitivity['gamma'].eq(1.0)].copy()
resolution_data = sensitivity.loc[(sensitivity['setting_type'].isin(['resolution', 'primary'])) & sensitivity['density'].eq(0.20)].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
sns.pointplot(data=density_data, x='density', y='mean_delta_q', hue='brain_area_label', palette=[palette[i] for i in (1, 2, 3)], errorbar=('ci', 95), dodge=0.12, markers='o', linestyles='-', ax=axes[0])
sns.pointplot(data=resolution_data, x='gamma', y='mean_delta_q', hue='brain_area_label', palette=[palette[i] for i in (1, 2, 3)], errorbar=('ci', 95), dodge=0.12, markers='o', linestyles='-', ax=axes[1])
axes[0].set_xlabel('Retained positive-edge density')
axes[1].set_xlabel(r'Leiden resolution $\gamma$')
axes[0].set_ylabel(r'Mean chance-corrected modularity ($\Delta Q$)')
axes[1].set_ylabel('')
axes[0].set_title(r'Density sensitivity ($\gamma=1$)')
axes[1].set_title('Resolution sensitivity (density = 0.20)')
axes[0].legend_.remove()
axes[1].legend(title='', frameon=False, loc='best')
for ax in axes:
    ax.axhline(0, color='0.45', linestyle='--', linewidth=1)
    sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(output_dir / 'q14_modularity_sensitivity.png', dpi=200, bbox_inches='tight')
plt.show()